In [1]:
%pip install -q -e "d:/Projects/queryforge"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Pipeline Input Parameters

In [2]:
from sagemaker.core.workflow.parameters import ParameterString, ParameterFloat

model_name          = ParameterString("ModelName",          default_value="Llama-3.2-1B-Instruct")
schema_name         = ParameterString("SchemaName",         default_value="orders")
schema_version      = ParameterString("SchemaVersion",      default_value="v1")
accuracy_threshold  = ParameterFloat("AccuracyThreshold",   default_value=0.75)

In [3]:
import importlib.util
import site
import sys
from pathlib import Path

from sagemaker.core.workflow.functions import Join
from queryforge.utils.config import ConfigLoader

config = ConfigLoader().load()

bucket = config.s3_bucket
prefix = config.s3_prefix

model_s3_uri = Join(
    on="/",
    values=[f"s3://{bucket}/{prefix}/models", model_name, schema_version],
)

dataset_s3_uri = Join(
    on="/",
    values=[f"s3://{bucket}/{prefix}/datasets", schema_name, schema_version],
)

output_s3_uri = Join(
    on="/",
    values=[f"s3://{bucket}/{prefix}/adapters", schema_name, schema_version],
)

eval_output_uri = Join(
    on="/",
    values=[f"s3://{bucket}/{prefix}/evaluation", schema_name, schema_version],
)

print("Model S3 URI:", model_s3_uri.to_string)


Model S3 URI: <bound method Join.to_string of Join(on='/', values=['s3://presmanes-queryforge-bucket/queryforge/models', ParameterString(name='ModelName', parameter_type=<ParameterTypeEnum.STRING: 'String'>, default_value='Llama-3.2-1B-Instruct'), ParameterString(name='SchemaVersion', parameter_type=<ParameterTypeEnum.STRING: 'String'>, default_value='v1')])>


# Fine Tuning Step

In [4]:
import os
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.train.model_trainer import ModelTrainer, Mode
from sagemaker.train.constants import TRAIN_SCRIPT
from sagemaker.train.configs import (
    Compute, InputData, OutputDataConfig, S3DataSource,
    SourceCode, StoppingCondition,
)

from sagemaker.mlops.workflow.steps import TrainingStep, ProcessingStep
from sagemaker.mlops.workflow.condition_step import ConditionStep
from sagemaker.mlops.workflow.pipeline import Pipeline

# Windows CRLF fix: open("w") writes \r\n on Windows; the Linux container
# rejects \r bytes with "$'\r': command not found".
_orig_prepare = ModelTrainer._prepare_train_script

def _prepare_train_script_lf(self, tmp_dir, source_code, distributed=None):
    _orig_prepare(self, tmp_dir, source_code, distributed)
    path = os.path.join(tmp_dir.name, TRAIN_SCRIPT)
    with open(path, "rb") as f:
        data = f.read()
    with open(path, "wb") as f:
        f.write(data.replace(b"\r\n", b"\n"))

ModelTrainer._prepare_train_script = _prepare_train_script_lf

pipeline_session = PipelineSession(
    boto_session=config.boto_session(),
    default_bucket=config.s3_bucket,
    default_bucket_prefix=config.s3_prefix,
)

hyperparameters = {
    "lora_r": config.train.lora_r,
    "lora_alpha": config.train.lora_alpha,
    "lora_dropout": config.train.lora_dropout,
    "epochs": config.train.epochs,
    "batch_size": config.train.batch_size,
    "grad_accum_steps": config.train.grad_accum_steps,
    "learning_rate": config.train.learning_rate,
    "max_seq_length": config.train.max_seq_length,
}

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\Presmanes\AppData\Local\sagemaker\sagemaker\config.yaml


[04/06/26 10:01:29] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=342643;file://d:\Projects\queryforge\.venv\lib\site-packages\botocore\credentials.py\credentials.py]8;;\:]8;id=801161;file://d:\Projects\queryforge\.venv\lib\site-packages\botocore\credentials.py#1392\1392]8;;\

In [5]:
from pathlib import Path

trainer = ModelTrainer(
    training_image      = config.processing_image_uri,  # Docker image URI for the training job
    sagemaker_session   = pipeline_session,             # Sagemaker Session for managing interactions with AWS services
    role                = config.execution_role_arn,    # IAM Role ARN with permissions for the training job
    source_code         = SourceCode(                   # Source code configuration for the training job
        source_dir=str(Path.cwd() / "train"),
        entry_script="train.py",
        requirements="requirements-train.txt",
    ),
    compute=Compute(
        instance_type=config.train.instance_type,
        instance_count=1,
    ),
    hyperparameters=hyperparameters,
    input_data_config=[
        InputData(channel_name="model",     data_source=S3DataSource(s3_uri=model_s3_uri,    s3_data_type="S3Prefix")),
        InputData(channel_name="training",  data_source=S3DataSource(s3_uri=dataset_s3_uri,  s3_data_type="S3Prefix")),
    ],
    output_data_config=OutputDataConfig(s3_output_path=output_s3_uri),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=config.train.max_runtime_seconds),
    base_job_name="queryforge-train",
    training_mode=Mode.SAGEMAKER_TRAINING_JOB,
)

training_step = TrainingStep(
    name="QLoraFineTune",
    step_args=trainer.train()
)

[04/06/26 10:01:30] INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=644729;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\train\defaults.py\defaults.py]8;;\:]8;id=136463;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\train\defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=506111;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\train\model_trainer.py\model_trainer.py]8;;\:]8;id=367080;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\train\model_trainer.py#553\553]8;;\
                             763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-                     
                             training:2.1.0-transformers4.36.0-gpu-py310-cu121-ubuntu20.04                         

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=303525;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\telemetry\telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=168179;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\telemetry\telemetry_logging.py#101\101]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


# Processing Step (Validation)

In [6]:
from sagemaker.core.processing import ScriptProcessor, ProcessingInput, ProcessingOutput, ProcessingS3Input
from sagemaker.core.shapes.shapes import ProcessingS3Output
from sagemaker.mlops.workflow.steps import ProcessingStep
from sagemaker.core.workflow.properties import PropertyFile

import os
os.environ["AWS_DEFAULT_REGION"] = config.aws_region

# Adapter URI viene del TrainingStep — referencia lazy resuelta en runtime
adapter_s3_uri = training_step.properties.ModelArtifacts.S3ModelArtifacts

validation_processor = ScriptProcessor(
    image_uri         = config.processing_image_uri,
    sagemaker_session = pipeline_session,
    command           = ["python3"],
    instance_type     = config.evaluation_instance_type,
    instance_count    = 1,
    role              = config.execution_role_arn,
    base_job_name     = "queryforge-evaluate",
)

metrics_output = PropertyFile(
    name        = "EvaluationMetrics",
    output_name = "metrics",
    path        = "metrics.json",
)

validation_step = ProcessingStep(
    name = "ModelEvaluation",
    step_args = validation_processor.run(
        code    = (Path.cwd() / "evaluate" / "evaluate.py").as_uri(),
        inputs  = [
            ProcessingInput(
                input_name = "model",
                s3_input   = ProcessingS3Input(
                    s3_uri       = model_s3_uri,
                    local_path   = "/opt/ml/processing/input/model",
                    s3_data_type = "S3Prefix",
                ),
            ),
            ProcessingInput(
                input_name = "adapter",
                s3_input   = ProcessingS3Input(
                    s3_uri       = adapter_s3_uri,
                    local_path   = "/opt/ml/processing/input/adapter",
                    s3_data_type = "S3Prefix",
                ),
            ),
            ProcessingInput(
                input_name = "dataset",
                s3_input   = ProcessingS3Input(
                    s3_uri       = dataset_s3_uri,
                    local_path   = "/opt/ml/processing/input/dataset",
                    s3_data_type = "S3Prefix",
                ),
            ),
        ],
        outputs = [
            ProcessingOutput(
                output_name = "metrics",
                s3_output   = ProcessingS3Output(
                    s3_uri          = eval_output_uri,
                    local_path      = "/opt/ml/processing/output",
                    s3_upload_mode  = "EndOfJob",
                ),
            ),
        ],
    ),
    property_files = [metrics_output],
)

[04/06/26 10:01:31] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=276890;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\telemetry\telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=898755;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\telemetry\telemetry_logging.py#101\101]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[04/06/26 10:01:32] WARNING  Windows Support for Local Mode is Experimental                    ]8;id=221352;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\local\local_session.py\local_session.py]8;;\:]8;id=509784;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\local\local_session.py#594\594]8;;\

# Condition Step (Evaluation Metrics)

In [7]:
from sagemaker.core.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.core.workflow.functions import JsonGet
from sagemaker.mlops.workflow.condition_step import ConditionStep

accuracy_check = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=validation_step.name,
        property_file=metrics_output,
        json_path="execution_accuracy",   # ← clave en metrics.json
    ),
    right=accuracy_threshold,
)

# Register Model Step

In [8]:
%pip install schema

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.mlops.workflow.model_step import ModelStep

model_builder = ModelBuilder(
    s3_model_data_url=adapter_s3_uri,
    image_uri=trainer.training_image,
    role_arn=config.execution_role_arn,
    sagemaker_session=pipeline_session,
    content_type="application/json",
    accept_type="application/json",
)

register_step = ModelStep(
    name="RegisterModel",
    step_args=model_builder.register(
        model_package_group_name=Join(on="-", values=["queryforge", schema_name]),
        approval_status="PendingManualApproval",
        content_types=["application/json"],
        response_types=["application/json"],
    ),
)

# Añádelo al if_steps del ConditionStep
condition_step = ConditionStep(
    name="AccuracyGate",
    conditions=[accuracy_check],
    if_steps=[register_step],
    else_steps=[],
)

[04/06/26 10:01:34] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=859744;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\serve\model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=541863;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\serve\model_builder_utils.py#337\337]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=201332;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\serve\model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=502038;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\serve\model_builder_utils.py#369\369]8;;\

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=743900;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\telemetry\telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=481461;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\telemetry\telemetry_logging.py#101\101]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

# Ensemble pipeline

In [10]:
pipeline = Pipeline(
    name="QueryForgeFinetuning",
    parameters=[model_name, schema_name, schema_version, accuracy_threshold],
    steps=[training_step, validation_step, condition_step],
    sagemaker_session=pipeline_session,
)

# Sube/actualiza la definición en AWS (no ejecuta nada todavía)
pipeline.upsert(
    role_arn=config.execution_role_arn,
    tags=[{"Key": "project", "Value": "queryforge"}],
)

# Ejecuta con los parámetros por defecto
execution = pipeline.start()

execution.wait()
# Opcional: esperar a que termine

[04/06/26 10:01:35] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=921926;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\telemetry\telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=375734;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\telemetry\telemetry_logging.py#101\101]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[04/06/26 10:01:39] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=289434;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py\utilities.py]8;;\:]8;id=531693;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

[04/06/26 10:01:40] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=598233;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py\utilities.py]8;;\:]8;id=987278;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition     ]8;id=984048;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\mlops\workflow\model_step.py\model_step.py]8;;\:]8;id=108815;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\mlops\workflow\model_step.py#195\195]8;;\
                             since it will be overridden in pipeline execution time.                               

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=168900;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py\utilities.py]8;;\:]8;id=201320;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[04/06/26 10:01:45] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=523958;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py\utilities.py]8;;\:]8;id=194323;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

[04/06/26 10:01:46] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=525131;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py\utilities.py]8;;\:]8;id=614766;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=908464;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py\utilities.py]8;;\:]8;id=359168;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\workflow\utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[04/06/26 10:01:47] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=422382;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\telemetry\telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=453881;file://d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\core\telemetry\telemetry_logging.py#101\101]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:17                                                                                   │
│                                                                                                  │
│   14 # Ejecuta con los parámetros por defecto                                                    │
│   15 execution = pipeline.start()                                                                │
│   16                                                                                             │
│ ❱ 17 execution.wait()                                                                            │
│   18 # Opcional: esperar a que termine                                                           │
│   19                                                                                             │
│                                                                                                  │
│ d:\Projects\queryforge\.venv\lib\site-packages\sagemaker\mlops\workflow\pipeline.py:1062 in wait │
│                                                                                                  │
│   1059 │   │   waiter = botocore.waiter.create_waiter_with_client(                               │
│   1060 │   │   │   waiter_id, model, self.sagemaker_session.sagemaker_client                     │
│   1061 │   │   )                                                                                 │
│ ❱ 1062 │   │   waiter.wait(PipelineExecutionArn=self.arn)                                        │
│   1063 │                                                                                         │
│   1064 │   def result(self, step_name: str):                                                     │
│   1065 │   │   """Retrieves the output of the provided step if it is a ``@step`` decorated func  │
│                                                                                                  │
│ d:\Projects\queryforge\.venv\lib\site-packages\botocore\waiter.py:58 in wait                     │
│                                                                                                  │
│    55 │   # Waiter.wait method. This is needed to attach a docstring to the                      │
│    56 │   # method.                                                                              │
│    57 │   def wait(self, **kwargs):                                                              │
│ ❱  58 │   │   Waiter.wait(self, **kwargs)                                                        │
│    59 │                                                                                          │
│    60 │   wait.__doc__ = WaiterDocstring(                                                        │
│    61 │   │   waiter_name=waiter_name,                                                           │
│                                                                                                  │
│ d:\Projects\queryforge\.venv\lib\site-packages\botocore\context.py:123 in wrapper                │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                            